# Paper Summarizer — section-wise, step by step
Run the cells in order. Each cell is one stage; re-run any stage after changing the model or file.
Outputs accumulate in `paper.notes` and are written to `report.md` at the end.

## 1. Load LLM

In [ ]:
import sys; sys.path.insert(0, "../src")   # not needed after `pip install -e .`
import ipywidgets as w
from papersum import LLM, Paper

backend = w.Dropdown(options=["ollama", "anthropic", "openai"], value="ollama", description="backend")
model   = w.Text(value="llama3.1:8b", description="model")   # e.g. qwen2.5:14b | claude-sonnet-4-6 | gpt-4o-mini
display(w.HBox([backend, model]))

In [ ]:
llm = LLM(backend.value, model.value)
print(llm.check())

## 2. Input paper (PDF / Markdown / EPUB)

In [ ]:
from pathlib import Path
upload = w.FileUpload(accept=".pdf,.md,.epub", multiple=False, description="Upload")
display(upload)

In [ ]:
# Option A - arXiv (preferred: real headings, tables and figures)
ARXIV = ""   # e.g. "1706.03762", a URL, or a title with load_arxiv(...)

from papersum.loaders import load_arxiv
if ARXIV:
    paper = Paper(llm).load_doc(load_arxiv(ARXIV))
    path = ARXIV.replace("/", "_")
else:
    paper = Paper(llm).load(path)   # Option B - the uploaded/local file


### 2b. Classify the outline

Sends every heading to the model in one call so subsections land in the right
category (`Optimizer` -> experiments, `Positional Encoding` -> method) instead of `other`.
Skip it to keep the offline regex classification.

In [ ]:
paper.classify_outline()

## 3. Retrieve related material (code repo / HF config / web) and plan the reading

In [ ]:
paper.fetch_related(use_web=True)   # set use_web=False when offline
paper.plan()

## 4. Research background and gap

In [ ]:
paper.background()

## 5. Motivation and method

In [ ]:
paper.method()

## 6. Workflow diagram (+ model layer diagram when code/config was found)

In [ ]:
paper.workflow_diagram()
paper.key_figures()          # picks the figures worth reproducing (arXiv only)
# paper.model_diagram()      # force layer-level diagram from paper text alone


## 7. Experiment results

In [ ]:
paper.results()

## 8. Limitations and next research gap

In [ ]:
paper.limitations()

## 9. Conclusion

In [ ]:
paper.conclusion()

## Export

In [ ]:
# arXiv ids look like file extensions to Path().stem, so prefer the id from metadata
stem = paper.doc.meta.get("id", "").replace("/", "_") or Path(path).stem
paper.export(f"../papers/reports/{stem}_report.md")
paper.export_html(f"../papers/reports/{stem}_report.html")   # one-page HTML
